In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
# Load MATLAB file
normal_bearings_data = loadmat('/content/drive/MyDrive/ML_Miniproject2/97.mat')
faulty_bearings_data1 = loadmat('/content/drive/MyDrive/ML_Miniproject2/105.mat')
faulty_bearings_data2 = loadmat('/content/drive/MyDrive/ML_Miniproject2/118.mat')
faulty_bearings_data3 = loadmat('/content/drive/MyDrive/ML_Miniproject2/130.mat')

normal_data = normal_bearings_data['X097_DE_time']
fault_data1 = faulty_bearings_data1['X105_DE_time']
fault_data2 = faulty_bearings_data2['X118_DE_time']
fault_data3 = faulty_bearings_data3['X130_DE_time']

def select_samples(data, M, N, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)

    samples = []
    for _ in range(M):
        start_index = np.random.randint(0, len(data) - N)
        sample = data[start_index:start_index + N]
        samples.append(sample)
    return np.array(samples)

M = 200
N = 300
normal_samples = select_samples(normal_data, M, N, 64)
faulty_samples1 = select_samples(fault_data1, M, N, 64)
faulty_samples2 = select_samples(fault_data2, M, N, 64)
faulty_samples3 = select_samples(fault_data3, M, N, 64)

labels_normal = np.zeros(M)
labels_faulty1 = np.ones(M)
labels_faulty2 = 2*np.ones(M)
labels_faulty3 = 3*np.ones(M)

normal_samples_flat = normal_samples.reshape(M, -1)
faulty_samples_flat1 = faulty_samples1.reshape(M, -1)
faulty_samples_flat2 = faulty_samples2.reshape(M, -1)
faulty_samples_flat3 = faulty_samples3.reshape(M, -1)

data_combined = np.vstack((normal_samples_flat, faulty_samples_flat1, faulty_samples_flat2, faulty_samples_flat3))

labels_combined = np.concatenate((labels_normal, labels_faulty1, labels_faulty2, labels_faulty3))

df = pd.DataFrame(data_combined)
df['label'] = labels_combined

In [ ]:
df

In [ ]:
def standard_deviation(x):
    return np.sqrt(np.sum((x - np.mean(x))**2) / len(x))

def peak(x):
    return np.max(np.abs(x))

def skewness(x):
    return np.sum((x - np.mean(x))**3) / (len(x) * standard_deviation(x)**3)

def kurtosis(x):
    return np.sum((x - np.mean(x))**4) / (len(x) * standard_deviation(x)**4)

def crest_factor(x):
    return peak(x) / np.sqrt(np.mean(x**2))

def clearance_factor(x):
    return peak(x) / np.mean(np.sqrt(np.abs(x)))

def peak_to_peak(x):
    return np.max(x) - np.min(x)

def shape_factor(x):
    return np.sqrt(np.mean(x**2)) / np.mean(np.abs(x))

def impact_factor(x):
    return peak(x) / np.mean(np.abs(x))

def square_mean_root(x):
    return np.sqrt(np.mean(np.sqrt(np.abs(x))))

def mean(x):
    return np.mean(x)

def absolute_mean(x):
    return np.mean(np.abs(x))

def root_mean_square(x):
    return np.sqrt(np.mean(x**2))

def impulse_factor(x):
    return peak(x) / root_mean_square(x)

feature_functions = {
    'Standard Deviation': standard_deviation,
    'Peak': peak,
    'Skewness': skewness,
    'Kurtosis': kurtosis,
    'Crest Factor': crest_factor,
    'Clearance Factor': clearance_factor,
    'Peak to Peak': peak_to_peak,
    'Shape Factor': shape_factor,
    'Impact Factor': impact_factor,
    'Square Mean Root': square_mean_root,
    'Mean': mean,
    'Absolute Mean': absolute_mean,
    'Root Mean Square': root_mean_square,
    'Impulse Factor': impulse_factor,
}


In [ ]:
data_features = df.iloc[:, :-1]
labels = df.iloc[:, -1]

extracted_features_df = pd.DataFrame()

for name, func in feature_functions.items():
    extracted_features_df[name] = data_features.apply(func, axis=1)

extracted_features_df['label'] = labels

In [ ]:
extracted_features_df

In [ ]:
X = extracted_features_df.drop(columns=['label'])
y = extracted_features_df['label']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_rem, y_train, y_rem = train_test_split(X, y,
                                                  train_size=0.6, random_state=4, shuffle=True)
X_valid, X_test, y_valid, y_test = train_test_split(X_rem,y_rem,
                                                    test_size=0.5, random_state=4, shuffle=True)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_normalized_data = scaler.fit_transform(X_train)
valid_normalized_data = scaler.transform(X_valid)
test_normalized_data = scaler.transform(X_test)

y_train = y_train.to_numpy()
y_valid = y_valid.to_numpy()
y_test = y_test.to_numpy()

In [ ]:
## Activation Function
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


## Loss
def bce(y, y_hat):
    return np.mean(-(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat)))

def mse(y, y_hat):
    return np.mean((y - y_hat)**2)

def categorical_cross_entropy(y, y_hat):
    return -np.mean(np.sum(y * np.log(y_hat), axis=1))


## Accuracy
def accuracy(y, y_hat, t=0.5):
    y_hat = np.where(y_hat < t, 0, 1)
    acc = np.sum(y == y_hat) / len(y)
    return acc

class MLP:
    def __init__(self, hidden_layer_sizes, hidden_activation='relu',
                 output_size=1, output_activation='sigmoid',
                 n_iter=1000, loss_fn=bce, eta=0.1, random_state=None):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.hidden_activation = hidden_activation
        self.output_size = output_size
        self.output_activation = output_activation
        self.n_iter = n_iter
        self.loss_fn = loss_fn
        self.eta = eta
        self.random_state = random_state
        np.random.seed(self.random_state)  # Set random seed

    def _init_weights(self):
        self.ws, self.bs = [], []  # Weight and bias lists for each layer
        self.as_ = [None] * len(self.hidden_layer_sizes)  # Initialize as_ with None
        all_layers = [self.input_size] + self.hidden_layer_sizes + [self.output_size]  # All layer sizes
        num_layers = len(all_layers)
        for i in range(1, num_layers):
            w = np.random.randn(all_layers[i-1], all_layers[i])  # Randomly initialize weights
            b = np.random.randn(all_layers[i])  # Randomly initialize biases
            self.ws.append(w)
            self.bs.append(b)

    def fit(self, X, y, X_val=None, y_val=None):  # Add optional validation data
        n, self.input_size = X.shape
        self._init_weights()
        train_losses = []
        val_losses = []
        train_accs = []
        val_accs = []
        for _ in range(self.n_iter):
            y_hat = self.predict(X)
            loss = self.loss_fn(y, y_hat)
            self._gradient_descent(X, y, y_hat)
            train_losses.append(loss)
            train_acc = accuracy(y, y_hat)
            train_accs.append(train_acc)
            if X_val is not None and y_val is not None:
                val_loss = self.loss_fn(y_val, self.predict(X_val))
                val_losses.append(val_loss)
                val_acc = accuracy(y_val, self.predict(X_val))
                val_accs.append(val_acc)
                print(f"Train Loss: {loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
            else:
                print(f"Train Loss: {loss:.4f} | Train Acc: {train_acc:.4f}")
        if X_val is not None and y_val is not None:
            self.plot_history(train_losses, val_losses, train_accs, val_accs)
        else:
            self.plot_history(train_losses, None, train_accs, None)


    def plot_history(self, train_losses, val_losses=None, train_accs=None, val_accs=None):
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.plot(train_losses, label='Training Loss')
        if val_losses is not None:
            plt.plot(val_losses, label='Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training and Validation Loss')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(train_accs, label='Training Accuracy')
        if val_accs is not None:
            plt.plot(val_accs, label='Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.title('Training and Validation Accuracy')
        plt.legend()

        plt.show()
    def _gradient_descent(self, X, y, y_hat):
        delta = y_hat - y  # Compute difference between predicted and true values
        for j in range(len(self.ws)-1, 0, -1):
            w_grad = (self.as_[j-1].T @ delta) / len(y)  # Compute weight gradient
            b_grad = delta.mean(0)  # Compute bias gradient
            self.ws[j] -= self.eta * w_grad  # Update weights
            self.bs[j] -= self.eta * b_grad  # Update biases
            delta = (delta @ self.ws[j].T) * (self._activation_derivative(self.hs[j-1], self.hidden_activation))

    def predict(self, X):
        self.hs = []  # Hidden layer outputs
        self.as_ = []  # Activation function outputs
        for i, (w, b) in enumerate(zip(self.ws[:-1], self.bs[:-1])):
            a = self.as_[i-1].copy() if i>0 else X.copy()  # Input to the hidden layer
            self.hs.append(a @ w + b)  # Compute hidden layer output
            self.as_.append(self._activation_function(self.hs[i], self.hidden_activation))  # Apply activation function
        y = self._activation_function(self.as_[-1] @ self.ws[-1] + self.bs[-1], self.output_activation)  # Output layer activation
        return y

    def _activation_function(self, x, activation):
        if activation == 'relu':
            return np.maximum(0, x)  # ReLU activation
        elif activation == 'sigmoid':
            return 1 / (1 + np.exp(-x))  # Sigmoid activation
        else:
            raise ValueError("Invalid activation function.")

    def _activation_derivative(self, x, activation):
        if activation == 'relu':
            return np.where(x > 0, 1, 0)  # Derivative of ReLU activation
        elif activation == 'sigmoid':
            sigmoid = self._activation_function(x, 'sigmoid')
            return sigmoid * (1 - sigmoid)  # Derivative of Sigmoid activation
        else:
            raise ValueError("Invalid activation function.")


In [ ]:
# mlp = MLP(hidden_layer_sizes=[6,4], hidden_activation='relu', output_size=4, output_activation='sigmoid', n_iter=2000, loss_fn=bce, eta=0.01, random_state=4)

mlp = MLP(hidden_layer_sizes=[16,12], hidden_activation='relu', output_size=4, output_activation='sigmoid', n_iter=2000, loss_fn=categorical_cross_entropy, eta=0.06, random_state=4)

# mlp = MLP(hidden_layer_sizes=[16,12], hidden_activation='sigmoid', output_size=4, output_activation='sigmoid', n_iter=2000, loss_fn=bce, eta=0.06, random_state=4)

# mlp = MLP(hidden_layer_sizes=[16,12], hidden_activation='relu', output_size=4, output_activation='sigmoid', n_iter=2000, loss_fn=mse, eta=0.06, random_state=4)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()

train_labels_onehot = encoder.fit_transform(y_train.reshape(-1, 1)).toarray()
validate_labels_onehot = encoder.transform(y_valid.reshape(-1, 1)).toarray()
test_labels_onehot = encoder.transform(y_test.reshape(-1, 1)).toarray()

In [ ]:
mlp.fit(train_normalized_data, train_labels_onehot, valid_normalized_data, validate_labels_onehot)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_hat = mlp.predict(test_normalized_data)
y_hat = np.argmax(y_hat, axis=1)
test_labels_onehot = np.argmax(test_labels_onehot, axis=1)
model_report = classification_report(test_labels_onehot, y_hat)
print(model_report)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


cm1 = confusion_matrix(test_labels_onehot, y_hat)

fig, ax = plt.subplots(1, 1, figsize=(15, 5))

sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted Labels')
ax.set_ylabel('True Labels')
ax.set_title('Confusion Matrix')

plt.show()

In [ ]:
mlp2 = MLP(hidden_layer_sizes=[16,12], hidden_activation='relu', output_size=4, output_activation='sigmoid', n_iter=2000, loss_fn=mse, eta=0.06, random_state=4)

In [ ]:
train_labels_onehot = encoder.fit_transform(y_train.reshape(-1, 1)).toarray()
validate_labels_onehot = encoder.transform(y_valid.reshape(-1, 1)).toarray()

In [ ]:
mlp2.fit(train_normalized_data, train_labels_onehot, valid_normalized_data, validate_labels_onehot)

In [ ]:
y_hat = mlp.predict(test_normalized_data)
y_hat = np.argmax(y_hat, axis=1)
# test_labels_onehot = np.argmax(test_labels_onehot, axis=1)
model_report = classification_report(test_labels_onehot, y_hat)
print(model_report)

In [ ]:
cm2 = confusion_matrix(test_labels_onehot, y_hat)

fig, ax = plt.subplots(1, 1, figsize=(15, 5))

sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted Labels')
ax.set_ylabel('True Labels')
ax.set_title('Confusion Matrix')

plt.show()

In [ ]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(solver='lbfgs', alpha=1e-5,
                    hidden_layer_sizes=(16, 12), random_state=4)
clf.fit(train_normalized_data, train_labels_onehot)

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(clf, train_normalized_data, train_labels_onehot,
                         cv=5, n_jobs=1)
mean_accuracy = scores.mean()

print("Mean Accuracy:", mean_accuracy)

In [ ]:
y_hat = clf.predict(test_normalized_data)
y_hat = np.argmax(y_hat, axis=1)
# test_labels_onehot = np.argmax(test_labels_onehot, axis=1)
model_report = classification_report(test_labels_onehot, y_hat)
print(model_report)

In [ ]:

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold
import numpy as np
import matplotlib.pyplot as plt

# Assuming MLP class and other utility functions (relu, sigmoid, etc.) are already defined as provided

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# One-hot encode the labels
# encoder = OneHotEncoder()
# y = y.to_numpy()
# y_onehot = encoder.fit_transform(y.reshape(-1, 1)).toarray()
# One-hot encode the labels
encoder = OneHotEncoder()
y = y.to_numpy()
y_onehot = encoder.fit_transform(y.reshape(-1, 1)).toarray()

# Define the model parameters
hidden_layer_sizes = [16, 12]
hidden_activation = 'relu'
output_size = y_onehot.shape[1]  # Number of classes
output_activation = 'sigmoid'
n_iter = 200
loss_fn = bce
eta = 0.06
random_state = 4

# Initialize K-Fold cross-validator
kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Lists to store performance metrics
train_accuracies = []
test_accuracies = []

for fold, (train_index, test_index) in enumerate(kf.split(X_scaled), 1):
    X_train_fold, X_test_fold = X_scaled[train_index], X_scaled[test_index]
    y_train_fold, y_test_fold = y_onehot[train_index], y_onehot[test_index]

    # Initialize the MLP model
    mlp_net = MLP(hidden_layer_sizes=hidden_layer_sizes, hidden_activation=hidden_activation,
                  output_size=output_size, output_activation=output_activation,
                  n_iter=n_iter, loss_fn=loss_fn, eta=eta, random_state=random_state)

    # Fit the model
    mlp_net.fit(X_train_fold, y_train_fold)

    # Evaluate on the training and test sets
    y_train_pred = mlp_net.predict(X_train_fold)
    y_test_pred = mlp_net.predict(X_test_fold)

    train_acc = accuracy(np.argmax(y_train_fold, axis=1), np.argmax(y_train_pred, axis=1))
    test_acc = accuracy(np.argmax(y_test_fold, axis=1), np.argmax(y_test_pred, axis=1))

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f"Fold {fold} - Training Accuracy: {train_acc:.4f}, Test Accuracy: {test_acc:.4f}")

# Average performance across all folds
avg_train_acc = np.mean(train_accuracies)
avg_test_acc = np.mean(test_accuracies)

print(f"Average Training Accuracy: {avg_train_acc:.4f}")
print(f"Average Test Accuracy: {avg_test_acc:.4f}")